# Причинно доступный индекс сопротивления бурению

Этот ноутбук анализирует энергоёмкость после обучения. Сначала выполните
`python train_advisory.py --backend lightgbm` (или явно выберите `sklearn`).
Обучение сохраняет калибровку, рассчитанную только по train-скважинам.
Категория текущей строки зависит от trailing-медианы 30 доступных наблюдений,
а не от медианы завершённого сегмента с будущими значениями.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import json
import pandas as pd
from drilling_advisory import prepare_history, EnergyCalibration
from config import RAW_DATA_PATH, DRILLING_ADVISORY_ARTIFACT_DIR
calibration_path = DRILLING_ADVISORY_ARTIFACT_DIR / "energy_calibration.json"
if not calibration_path.exists():
    raise RuntimeError("Сначала запустите train_advisory.py для калибровки на train-скважинах")
calibration = EnergyCalibration.from_dict(json.loads(calibration_path.read_text(encoding="utf-8")))
history = prepare_history(pd.read_csv(RAW_DATA_PATH))
df = calibration.transform(history)


In [ ]:
display(df.groupby("rock_energy_type_final").agg(
    rows=("speed", "size"), speed_median=("speed", "median"),
    hardness_median=("hardness_score_smooth", "median")))
display(pd.read_csv(PROJECT_ROOT / "notebooks/model_reports/near5_energy_offline_evaluation.csv"))


`pseudo_mse` — условный телеметрический индекс, не физическая MSE и не
независимо измеренная крепость породы. Глобальная 3D-поверхность симулятора —
эмпирическая визуализация по train-строкам; рекомендация считается отдельно моделями.
